# Heart Age Engine — Reproducible Analysis
This notebook is a short walkthrough of the public refactor. The core logic lives in `src/heart_age_engine/`; this notebook calls those modules instead of duplicating the implementation.

> Educational project only. Outputs are model-derived estimates, not medical diagnoses.

In [ ]:
from heart_age_engine.anomaly_detection import find_label_model_mismatches, run_lof
from heart_age_engine.clustering import cluster_summary, fit_clusters
from heart_age_engine.data import load_and_prepare_data
from heart_age_engine.explainability import global_importance
from heart_age_engine.modeling import (
    add_heart_age_predictions,
    compare_feature_sets,
    train_heart_age_model,
)


## 1. Load and clean the public dataset

In [ ]:
DATA_PATH = '../data/cardio_train.xlsx'  # or cardio_train.csv
df = load_and_prepare_data(DATA_PATH)
df[['age_years', 'bmi', 'pulse_pressure', 'cardio']].describe().round(2)


## 2. Train a leakage-safe heart-age proxy
Chronological age is the target and is deliberately excluded from the predictors.

In [ ]:
model = train_heart_age_model(df)
print(f'MAE: {model.mae:.2f} years')
print(f'R²: {model.r2:.3f}')
scored = add_heart_age_predictions(df, model)


## 3. Compare feature subsets

In [ ]:
compare_feature_sets(df)

## 4. Explain the fitted model with SHAP

In [ ]:
global_importance(model, scored)

## 5. Local Outlier Factor
`contamination=0.05` is a chosen screening threshold, not an estimate of clinical prevalence.

In [ ]:
lof = run_lof(scored, contamination=0.05)
print('Flagged:', (lof.frame.lof_label == -1).sum())


## 6. Explore label/model disagreements

In [ ]:
high_gap_healthy, low_gap_disease = find_label_model_mismatches(lof.frame)
print('High-gap, healthy-label cases:', len(high_gap_healthy))
print('Low-gap, disease-label cases:', len(low_gap_disease))


## 7. K-Means exploratory profiles

In [ ]:
clustered, cluster_model = fit_clusters(scored, n_clusters=8)
cluster_summary(clustered)
